# 🎓 The Dropout Signal — Medallion Pipeline
## Bronze → Silver → Gold | HackBricks 2026

**Problem:** Predict university dropout risk early, fairly, and explainably.

**This notebook:**
1. **Bronze** — Ingest `uci_dropout.csv` → raw Delta table
2. **Silver** — Clean nulls, engineer features (grade trajectory, financial stress index, etc.)
3. **Gold** — Created in the ML notebook after model scoring

---
**Dataset:** `/Volumes/workspace/default/raw/uci_dropout.csv` | 4,400 records | 37 features

In [ ]:
# ============================================================================
#  Cell 1: Configuration & Imports
# ============================================================================
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType, StringType
from datetime import datetime

# ── Config ───────────────────────────────────────────────────────────
CATALOG   = "workspace"
SCHEMA    = "default"
RAW_PATH  = "/Volumes/workspace/default/raw/uci_dropout.csv"

# Table names
BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.dropout_bronze"
SILVER_TABLE = f"{CATALOG}.{SCHEMA}.dropout_silver"
GOLD_TABLE   = f"{CATALOG}.{SCHEMA}.dropout_gold"

# Create the volume if it doesn't exist
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.raw")

print(f"📁 Source:  {RAW_PATH}")
print(f"🥉 Bronze:  {BRONZE_TABLE}")
print(f"🥈 Silver:  {SILVER_TABLE}")
print(f"🥇 Gold:    {GOLD_TABLE}")

---
## 🥉 BRONZE LAYER — Raw Ingestion
Read the UCI Dropout CSV **as-is** into a Delta table. No transformations — just a faithful record of the source data with ingestion metadata.

In [ ]:
# ============================================================================
#  Cell 2: BRONZE — Ingest raw CSV into Delta table
# ============================================================================

# Read CSV with header and infer schema
raw_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("sep", ";")          # UCI dropout dataset uses semicolon delimiter
    .csv(RAW_PATH)
)

# Standardize column names: replace spaces/special chars with underscores
for col_name in raw_df.columns:
    clean_name = (
        col_name.strip()
        .replace(" ", "_")
        .replace("(", "")
        .replace(")", "")
        .replace("'", "")
        .replace("/", "_")
        .lower()
    )
    raw_df = raw_df.withColumnRenamed(col_name, clean_name)

# Add ingestion metadata
bronze_df = (
    raw_df
    .withColumn("_ingestion_ts", F.current_timestamp())
    .withColumn("_source_file", F.lit(RAW_PATH))
)

# Write Bronze Delta table
bronze_df.write.format("delta").mode("overwrite").saveAsTable(BRONZE_TABLE)

row_count = spark.table(BRONZE_TABLE).count()
col_count = len(spark.table(BRONZE_TABLE).columns)
print(f"✅ Bronze table created: {BRONZE_TABLE}")
print(f"   Rows: {row_count:,} | Columns: {col_count}")

In [ ]:
# ============================================================================
#  Cell 3: BRONZE — Schema Documentation
# ============================================================================

bronze = spark.table(BRONZE_TABLE)

print("=" * 70)
print("  BRONZE SCHEMA DOCUMENTATION")
print("=" * 70)
print(f"  Table:   {BRONZE_TABLE}")
print(f"  Rows:    {bronze.count():,}")
print(f"  Columns: {len(bronze.columns)}")
print(f"  Format:  Delta Lake")
print("=" * 70)
print()

# Print schema with data types
for i, field in enumerate(bronze.schema.fields, 1):
    if field.name.startswith("_"):
        tag = "(metadata)"
    elif field.name == "target":
        tag = "(TARGET VARIABLE)"
    else:
        tag = ""
    print(f"  {i:>2}. {field.name:<55} {str(field.dataType):<15} {tag}")

# Show target distribution
print("\n" + "─" * 70)
print("  TARGET DISTRIBUTION:")
print("─" * 70)
display(
    bronze.groupBy("target")
    .count()
    .withColumn("pct", F.round(F.col("count") / bronze.count() * 100, 1))
    .orderBy("target")
)

---
## 🥈 SILVER LAYER — Clean + Feature Engineering

Transformations:
1. **Binary target**: Dropout=1, Enrolled/Graduate=0
2. **Clean nulls** & validate ranges
3. **6 Engineered features:**
   - `grade_trajectory` — slope of Sem1→Sem2 avg grades
   - `financial_stress_index` — composite of tuition + debtor + scholarship + macro
   - `curricular_engagement_sem1/sem2` — approved/enrolled ratio
   - `semester_grade_delta` — Sem2 avg grade - Sem1 avg grade
   - `economic_pressure_index` — unemployment × inflation / GDP
   - `age_risk_flag` — binary flag for mature students (>25)

In [ ]:
# ============================================================================
#  Cell 4: SILVER — Clean + Feature Engineering
# ============================================================================

bronze = spark.table(BRONZE_TABLE)

# ── Step 1: Binary target ────────────────────────────────────────────
# Dropout = 1, Enrolled/Graduate = 0
silver = bronze.withColumn(
    "is_dropout",
    F.when(F.lower(F.col("target")) == "dropout", 1).otherwise(0)
)

# ── Step 2: Clean nulls — fill numeric nulls with 0 ─────────────────
numeric_cols = [
    f.name for f in silver.schema.fields
    if isinstance(f.dataType, (DoubleType, IntegerType))
    and not f.name.startswith("_")
]
silver = silver.fillna(0, subset=numeric_cols)

# ── Step 3: Compute average grades per semester ──────────────────────
# Find all Sem1 and Sem2 grade columns dynamically
sem1_grade_cols = [c for c in silver.columns if "1st_sem" in c and "grade" in c]
sem2_grade_cols = [c for c in silver.columns if "2nd_sem" in c and "grade" in c]

# Average grade across all sem1/sem2 subjects
if sem1_grade_cols:
    silver = silver.withColumn(
        "avg_grade_sem1",
        sum(F.col(c) for c in sem1_grade_cols) / len(sem1_grade_cols)
    )
else:
    silver = silver.withColumn("avg_grade_sem1", F.lit(0.0))

if sem2_grade_cols:
    silver = silver.withColumn(
        "avg_grade_sem2",
        sum(F.col(c) for c in sem2_grade_cols) / len(sem2_grade_cols)
    )
else:
    silver = silver.withColumn("avg_grade_sem2", F.lit(0.0))

# ── FEATURE 1: Grade Trajectory (slope) ─────────────────────────────
# Positive = improving, Negative = declining
silver = silver.withColumn(
    "grade_trajectory",
    F.round(F.col("avg_grade_sem2") - F.col("avg_grade_sem1"), 4)
)

# ── FEATURE 2: Financial Stress Index ────────────────────────────────
# Composite: tuition + debtor + no-scholarship + macro pressure
# Tuition: column likely "tuition_fees_up_to_date" → 1=paid, 0=unpaid
tuition_col = [c for c in silver.columns if "tuition" in c]
debtor_col = [c for c in silver.columns if "debtor" in c]
scholarship_col = [c for c in silver.columns if "scholarship" in c]
inflation_col = [c for c in silver.columns if "inflation" in c]
gdp_col = [c for c in silver.columns if "gdp" in c]

# Build financial stress — higher = more stressed
tuition_stress = (1 - F.col(tuition_col[0]).cast("double")) * 0.35 if tuition_col else F.lit(0.0)
debtor_stress = F.col(debtor_col[0]).cast("double") * 0.25 if debtor_col else F.lit(0.0)
no_scholarship = (1 - F.col(scholarship_col[0]).cast("double")) * 0.20 if scholarship_col else F.lit(0.0)

# Normalize inflation (typical range 0-5%) and GDP (typical range -3 to 5%)
inflation_norm = F.col(inflation_col[0]).cast("double") / 5.0 * 0.10 if inflation_col else F.lit(0.0)
gdp_norm = (1 - F.col(gdp_col[0]).cast("double") / 5.0) * 0.10 if gdp_col else F.lit(0.0)

silver = silver.withColumn(
    "financial_stress_index",
    F.round(tuition_stress + debtor_stress + no_scholarship + inflation_norm + gdp_norm, 4)
)

# ── FEATURE 3: Curricular Engagement Ratio (per semester) ────────────
sem1_approved = [c for c in silver.columns if "1st_sem" in c and "approved" in c]
sem1_enrolled = [c for c in silver.columns if "1st_sem" in c and "enrolled" in c]
sem2_approved = [c for c in silver.columns if "2nd_sem" in c and "approved" in c]
sem2_enrolled = [c for c in silver.columns if "2nd_sem" in c and "enrolled" in c]

if sem1_approved and sem1_enrolled:
    silver = silver.withColumn(
        "curricular_engagement_sem1",
        F.round(
            F.when(F.col(sem1_enrolled[0]) > 0,
                   F.col(sem1_approved[0]) / F.col(sem1_enrolled[0]))
            .otherwise(0.0), 4
        )
    )
else:
    silver = silver.withColumn("curricular_engagement_sem1", F.lit(0.0))

if sem2_approved and sem2_enrolled:
    silver = silver.withColumn(
        "curricular_engagement_sem2",
        F.round(
            F.when(F.col(sem2_enrolled[0]) > 0,
                   F.col(sem2_approved[0]) / F.col(sem2_enrolled[0]))
            .otherwise(0.0), 4
        )
    )
else:
    silver = silver.withColumn("curricular_engagement_sem2", F.lit(0.0))

# ── FEATURE 4: Semester Grade Delta ──────────────────────────────────
silver = silver.withColumn(
    "semester_grade_delta",
    F.round(F.col("avg_grade_sem2") - F.col("avg_grade_sem1"), 4)
)

# ── FEATURE 5: Economic Pressure Index ───────────────────────────────
unemployment_col = [c for c in silver.columns if "unemployment" in c]

if unemployment_col and inflation_col and gdp_col:
    silver = silver.withColumn(
        "economic_pressure_index",
        F.round(
            F.col(unemployment_col[0]).cast("double")
            * F.col(inflation_col[0]).cast("double")
            / F.greatest(F.abs(F.col(gdp_col[0]).cast("double")), F.lit(0.1)),
            4
        )
    )
else:
    silver = silver.withColumn("economic_pressure_index", F.lit(0.0))

# ── FEATURE 6: Age Risk Flag ────────────────────────────────────────
age_col = [c for c in silver.columns if "age" in c and "enrollment" in c]
if age_col:
    silver = silver.withColumn(
        "age_risk_flag",
        F.when(F.col(age_col[0]) > 25, 1).otherwise(0)
    )
else:
    silver = silver.withColumn("age_risk_flag", F.lit(0))

# ── Drop metadata columns before saving ──────────────────────────────
silver = silver.drop("_ingestion_ts", "_source_file")

# ── Write Silver Delta table ─────────────────────────────────────────
silver.write.format("delta").mode("overwrite").saveAsTable(SILVER_TABLE)

row_count = spark.table(SILVER_TABLE).count()
col_count = len(spark.table(SILVER_TABLE).columns)
print(f"✅ Silver table created: {SILVER_TABLE}")
print(f"   Rows: {row_count:,} | Columns: {col_count}")
print(f"\n   Engineered features added:")
print(f"   ├── grade_trajectory")
print(f"   ├── financial_stress_index")
print(f"   ├── curricular_engagement_sem1")
print(f"   ├── curricular_engagement_sem2")
print(f"   ├── semester_grade_delta")
print(f"   ├── economic_pressure_index")
print(f"   └── age_risk_flag")

In [ ]:
# ============================================================================
#  Cell 5: SILVER — Validate Engineered Features
# ============================================================================

silver = spark.table(SILVER_TABLE)

print("=" * 70)
print("  SILVER LAYER — Feature Engineering Validation")
print("=" * 70)

# Target distribution (binary)
print("\n📊 Binary Target Distribution:")
display(
    silver.groupBy("is_dropout")
    .count()
    .withColumn("label", F.when(F.col("is_dropout") == 1, "Dropout").otherwise("Enrolled/Graduate"))
    .withColumn("pct", F.round(F.col("count") / silver.count() * 100, 1))
    .select("label", "is_dropout", "count", "pct")
    .orderBy("is_dropout")
)

# Engineered feature stats
eng_features = [
    "grade_trajectory", "financial_stress_index",
    "curricular_engagement_sem1", "curricular_engagement_sem2",
    "semester_grade_delta", "economic_pressure_index", "age_risk_flag"
]

print("\n📈 Engineered Feature Statistics (by dropout status):")
for feat in eng_features:
    if feat in silver.columns:
        stats = silver.groupBy("is_dropout").agg(
            F.round(F.avg(feat), 3).alias("mean"),
            F.round(F.stddev(feat), 3).alias("std"),
            F.round(F.min(feat), 3).alias("min"),
            F.round(F.max(feat), 3).alias("max"),
        )
        print(f"\n  ── {feat} ──")
        display(stats.orderBy("is_dropout"))

---
## ✅ Pipeline Complete

| Layer | Table | Status |
|-------|-------|--------|
| 🥉 Bronze | `dropout_bronze` | Raw CSV → Delta (all 37 original columns + metadata) |
| 🥈 Silver | `dropout_silver` | Cleaned + 7 engineered features + binary target |
| 🥇 Gold | `dropout_gold` | Created in `dropout_ml.ipynb` after model scoring |

**Next:** Open `dropout_ml.ipynb` to train models, run fairness audit, SHAP, and create the Gold table.